In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

base = Path.cwd()

if not (base / "data" / "processed").is_dir():
    base = base.parent

input_path = base / "data" / "processed" / "matches_clean.csv"
output_dir = base / "data" / "processed"

if not input_path.is_file():
    raise FileNotFoundError(
        f"Missing {input_path}. Run notebook 01 first."
    )

In [2]:
matches = pd.read_csv(
    input_path,
    dtype={
        "match_id": "string",
        "league": "string",
        "team_a": "string",
        "team_b": "string",
    },
)

required_columns = {
    "match_id",
    "date",
    "league",
    "team_a",
    "team_b",
    "result_a",
}

missing_columns = required_columns - set(matches.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

if matches[list(required_columns)].isna().any().any():
    raise ValueError("Required fields contain missing values.")

if not matches["match_id"].is_unique:
    raise ValueError("Duplicate match IDs found.")

matches["date"] = pd.to_datetime(
    matches["date"],
    utc=True,
    errors="raise",
    format="mixed",
)

competition_summary = (
    matches.groupby("league")
    .agg(
        games=("match_id", "size"),
        first_game=("date", "min"),
        last_game=("date", "max"),
    )
    .sort_values("games", ascending=False)
    .reset_index()
)

print("Competitions:", len(competition_summary))
display(competition_summary)

Competitions: 124


,league,games,first_game,last_game
0,LDL,7019,2018-03-03 06:15:12+00:00,2024-06-05 11:24:34+00:00
1,LPL,6918,2017-01-19 09:45:22+00:00,2026-09-13 12:03:56+00:00
2,LCK,4978,2017-01-17 07:10:23+00:00,2026-09-13 08:19:05+00:00
3,LCKC,2664,2021-01-18 07:53:00+00:00,2026-09-15 09:47:51+00:00
4,VCS,2526,2018-01-18 08:41:54+00:00,2026-10-11 12:15:18+00:00
...,...,...,...,...
119,MSC,25,2020-05-28 04:36:28+00:00,2020-05-31 11:41:52+00:00
120,Riot,20,2020-05-27 08:46:58+00:00,2020-05-30 12:37:45+00:00
121,AC,19,2026-03-04 21:12:25+00:00,2026-03-08 20:51:01+00:00
122,LGL,8,2019-04-06 11:43:13+00:00,2019-04-07 13:18:53+00:00


In [10]:
DOMESTIC_POOLS = {
    "LCK": "LCK",
    "LPL": "LPL",

    # Explicit continuity assumptions for renamed competitions.
    "LEC": "LEC",
    "EU LCS": "LEC",
    "LCS": "LCS",
    "NA LCS": "LCS",

    "PCS": "PCS",
    "VCS": "VCS",
    "CBLOL": "CBLOL",
    "LLA": "LLA",
    "LJL": "LJL",
    "TCL": "TCL",
    "LCO": "LCO",

    # Keep these histories separate initially.
    "LMS": "LMS",
    "OPL": "OPL",
    "LCL": "LCL",
    "LCP": "LCP",
    "LTA N": "LTA_N",
    "LTA S": "LTA_S",
}

In [11]:
INTERNATIONAL_COMPETITIONS = {
    "MSI",
    "WLDs",
    "FST",
}

# Regional bridge events: useful for connecting rating pools,
# but participation must not establish a new domestic home.
CROSS_POOL_COMPETITIONS = {
    "LTA",
}

# Excluded from regional learning in this first version.
# These are not assigned to their parent top-level rating pools.
DEVELOPMENTAL_COMPETITIONS = {
    "LDL",
    "LCKC",
    "CK",
    "LSPL",
    "LCSA",
    "NACL",
    "CBLOLA",
    "LJLA",
    "EU CS",
    "NA CS",
    "LJLCS",
    "OCS",
    "BRCC",
}

groups = {
    "domestic": set(DOMESTIC_POOLS),
    "international": INTERNATIONAL_COMPETITIONS,
    "cross_pool": CROSS_POOL_COMPETITIONS,
    "developmental": DEVELOPMENTAL_COMPETITIONS,
}

group_names = list(groups)

for index, first in enumerate(group_names):
    for second in group_names[index + 1:]:
        overlap = groups[first] & groups[second]

        if overlap:
            raise ValueError(
                f"Overlapping classifications: "
                f"{first}, {second}: {overlap}"
            )

In [12]:
classification = competition_summary.copy()

classification["competition_type"] = "review"

classification["rating_pool"] = pd.Series(
    pd.NA,
    index=classification.index,
    dtype="string",
)

classification["updates_home_pool"] = False
classification["use_local_elo"] = False
classification["use_region_calibration"] = False
classification["use_international_model"] = False

classification["classification_note"] = (
    "Not reviewed; excluded from regional learning."
)

# Domestic competitions.
domestic_mask = classification["league"].isin(DOMESTIC_POOLS)

classification.loc[
    domestic_mask, "competition_type"
] = "domestic"

classification.loc[
    domestic_mask, "rating_pool"
] = classification.loc[
    domestic_mask, "league"
].map(DOMESTIC_POOLS)

classification.loc[
    domestic_mask, "updates_home_pool"
] = True

classification.loc[
    domestic_mask, "use_local_elo"
] = True

classification.loc[
    domestic_mask, "classification_note"
] = "Included domestic rating pool."

# Global international events.
international_mask = classification["league"].isin(
    INTERNATIONAL_COMPETITIONS
)

classification.loc[
    international_mask, "competition_type"
] = "international"

classification.loc[
    international_mask, "use_region_calibration"
] = True

classification.loc[
    international_mask, "use_international_model"
] = True

classification.loc[
    international_mask, "classification_note"
] = "Global international event; does not update home pool."

# Cross-pool regional events.
cross_pool_mask = classification["league"].isin(
    CROSS_POOL_COMPETITIONS
)

classification.loc[
    cross_pool_mask, "competition_type"
] = "cross_pool"

classification.loc[
    cross_pool_mask, "use_region_calibration"
] = True

classification.loc[
    cross_pool_mask, "classification_note"
] = (
    "Connects historical home pools; excluded from the "
    "global international model in this version."
)

# Developmental competitions.
developmental_mask = classification["league"].isin(
    DEVELOPMENTAL_COMPETITIONS
)

classification.loc[
    developmental_mask, "competition_type"
] = "developmental"

classification.loc[
    developmental_mask, "classification_note"
] = (
    "Retained, but excluded from regional learning "
    "in this version."
)

display(classification)

,league,games,first_game,last_game,competition_type,rating_pool,updates_home_pool,use_local_elo,use_region_calibration,use_international_model,classification_note
0,LDL,7019,2018-03-03 06:15:12+00:00,2024-06-05 11:24:34+00:00,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
1,LPL,6918,2017-01-19 09:45:22+00:00,2026-09-13 12:03:56+00:00,domestic,LPL,True,True,False,False,Included domestic rating pool.
2,LCK,4978,2017-01-17 07:10:23+00:00,2026-09-13 08:19:05+00:00,domestic,LCK,True,True,False,False,Included domestic rating pool.
3,LCKC,2664,2021-01-18 07:53:00+00:00,2026-09-15 09:47:51+00:00,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
4,VCS,2526,2018-01-18 08:41:54+00:00,2026-10-11 12:15:18+00:00,domestic,VCS,True,True,False,False,Included domestic rating pool.
...,...,...,...,...,...,...,...,...,...,...,...
119,MSC,25,2020-05-28 04:36:28+00:00,2020-05-31 11:41:52+00:00,review,<NA>,False,False,False,False,Not reviewed; excluded from regional learning.
120,Riot,20,2020-05-27 08:46:58+00:00,2020-05-30 12:37:45+00:00,review,<NA>,False,False,False,False,Not reviewed; excluded from regional learning.
121,AC,19,2026-03-04 21:12:25+00:00,2026-03-08 20:51:01+00:00,review,<NA>,False,False,False,False,Not reviewed; excluded from regional learning.
122,LGL,8,2019-04-06 11:43:13+00:00,2019-04-07 13:18:53+00:00,review,<NA>,False,False,False,False,Not reviewed; excluded from regional learning.


In [13]:
assert classification["league"].is_unique

boolean_columns = [
    "updates_home_pool",
    "use_local_elo",
    "use_region_calibration",
    "use_international_model",
]

assert classification[boolean_columns].notna().all().all()

home_updates = classification["updates_home_pool"]

assert classification.loc[
    home_updates, "rating_pool"
].notna().all()

assert classification.loc[
    home_updates, "competition_type"
].eq("domestic").all()

assert classification.loc[
    classification["use_local_elo"],
    "rating_pool",
].notna().all()

assert not classification.loc[
    classification["competition_type"].isin(
        ["international", "cross_pool", "developmental", "review"]
    ),
    "updates_home_pool",
].any()

print("Classification checks passed.")

Classification checks passed.


In [6]:
classified_matches = matches.merge(
    classification[
        ["league", "competition_type", "rating_pool"]
    ],
    on="league",
    how="left",
    validate="many_to_one",
)

assert len(classified_matches) == len(matches)
assert classified_matches["competition_type"].notna().all()

domestic_matches = classified_matches.loc[
    classified_matches["competition_type"].eq("domestic")
]

assert domestic_matches["rating_pool"].notna().all()

international_matches = classified_matches.loc[
    classified_matches["competition_type"].eq("international")
]

assert international_matches["rating_pool"].isna().all()

display(classified_matches.head(10))

,match_id,date,league,team_a,team_b,result_a,competition_type,rating_pool
0,2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056,2017-01-10 15:44:06+00:00,EU CS,EURONICS Gaming,Fnatic Academy,0,review,<NA>
1,2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066,2017-01-10 16:21:23+00:00,EU CS,AlienTech eSports,Team Larssen,0,review,<NA>
2,2017-01-10 17:01:50+00:00|EU CS|TREU/1001490072,2017-01-10 17:01:50+00:00,EU CS,Bobaski,Nerv,0,review,<NA>
3,2017-01-10 17:55:36+00:00|EU CS|TREU/1001490081,2017-01-10 17:55:36+00:00,EU CS,Team Forge,Tricked Esport,1,review,<NA>
4,2017-01-10 18:57:51+00:00|EU CS|TREU/1001490084,2017-01-10 18:57:51+00:00,EU CS,ASUS ROG Army,LDLC OL,0,review,<NA>
5,2017-01-11 15:33:10+00:00|EU CS|TREU/1001490139,2017-01-11 15:33:10+00:00,EU CS,Fnatic Academy,LDLC OL,1,review,<NA>
6,2017-01-11 15:46:32+00:00|EU CS|TREU/1001490141,2017-01-11 15:46:32+00:00,EU CS,ALTERNATE aTTaX,AlienTech eSports,0,review,<NA>
7,2017-01-11 16:44:20+00:00|EU CS|TREU/1001490143,2017-01-11 16:44:20+00:00,EU CS,ASUS ROG Army,Nerv,0,review,<NA>
8,2017-01-11 17:53:03+00:00|EU CS|TREU/1001490147,2017-01-11 17:53:03+00:00,EU CS,Tricked Esport,devils.one inStreamly,0,review,<NA>
9,2017-01-11 18:59:33+00:00|EU CS|TREU/1001490149,2017-01-11 18:59:33+00:00,EU CS,Bobaski,EURONICS Gaming,0,review,<NA>


In [14]:
coverage = (
    classification.groupby("competition_type")
    .agg(
        competitions=("league", "size"),
        games=("games", "sum"),
    )
)

coverage["game_share"] = (
    coverage["games"] / coverage["games"].sum()
)

display(coverage)

print("Domestic rating pools:")
display(
    classification.loc[
        classification["competition_type"].eq("domestic"),
        ["league", "rating_pool", "games"],
    ]
)

print("Games eligible for each learning component:")

for flag in boolean_columns:
    game_count = classification.loc[ classification[flag], "games"].sum()
    print(f"{flag}: {game_count:,}")

,competitions,games,game_share
competition_type,,,
cross_pool,1,40,0.000416
developmental,13,17307,0.180141
domestic,19,33616,0.349893
international,3,1952,0.020317
review,88,43160,0.449232


Domestic rating pools:


,league,rating_pool,games
1,LPL,LPL,6918
2,LCK,LCK,4978
4,VCS,VCS,2526
5,LEC,LEC,2267
6,TCL,TCL,2258
7,LJL,LJL,2206
9,CBLOL,CBLOL,2005
11,LCS,LCS,1918
14,PCS,PCS,1786
25,LLA,LLA,1153


Games eligible for each learning component:
updates_home_pool: 33,616
use_local_elo: 33,616
use_region_calibration: 1,992
use_international_model: 1,952


In [15]:
mapping_columns = [
    "league",
    "competition_type",
    "rating_pool",
    *boolean_columns,
    "classification_note",
]

classified_matches = matches.merge(
    classification[mapping_columns],
    on="league",
    how="left",
    validate="many_to_one",
)

assert len(classified_matches) == len(matches)
assert classified_matches["match_id"].is_unique
assert classified_matches["competition_type"].notna().all()

classified_matches = (
    classified_matches
    .sort_values(["date", "match_id"])
    .reset_index(drop=True)
)

display(classified_matches.head(10))

,match_id,date,league,team_a,team_b,result_a,competition_type,rating_pool,updates_home_pool,use_local_elo,use_region_calibration,use_international_model,classification_note
0,2017-01-10 15:44:06+00:00|EU CS|TREU/1001490056,2017-01-10 15:44:06+00:00,EU CS,EURONICS Gaming,Fnatic Academy,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
1,2017-01-10 16:21:23+00:00|EU CS|TREU/1001490066,2017-01-10 16:21:23+00:00,EU CS,AlienTech eSports,Team Larssen,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
2,2017-01-10 17:01:50+00:00|EU CS|TREU/1001490072,2017-01-10 17:01:50+00:00,EU CS,Bobaski,Nerv,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
3,2017-01-10 17:55:36+00:00|EU CS|TREU/1001490081,2017-01-10 17:55:36+00:00,EU CS,Team Forge,Tricked Esport,1,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
4,2017-01-10 18:57:51+00:00|EU CS|TREU/1001490084,2017-01-10 18:57:51+00:00,EU CS,ASUS ROG Army,LDLC OL,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
5,2017-01-11 15:33:10+00:00|EU CS|TREU/1001490139,2017-01-11 15:33:10+00:00,EU CS,Fnatic Academy,LDLC OL,1,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
6,2017-01-11 15:46:32+00:00|EU CS|TREU/1001490141,2017-01-11 15:46:32+00:00,EU CS,ALTERNATE aTTaX,AlienTech eSports,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
7,2017-01-11 16:44:20+00:00|EU CS|TREU/1001490143,2017-01-11 16:44:20+00:00,EU CS,ASUS ROG Army,Nerv,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
8,2017-01-11 17:53:03+00:00|EU CS|TREU/1001490147,2017-01-11 17:53:03+00:00,EU CS,Tricked Esport,devils.one inStreamly,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."
9,2017-01-11 18:59:33+00:00|EU CS|TREU/1001490149,2017-01-11 18:59:33+00:00,EU CS,Bobaski,EURONICS Gaming,0,developmental,<NA>,False,False,False,False,"Retained, but excluded from regional learning ..."


In [16]:
output_dir.mkdir(parents=True, exist_ok=True)

classification.to_csv(
    output_dir / "competition_classification.csv",
    index=False,
)

classified_matches.to_csv(
    output_dir / "matches_classified.csv",
    index=False,
)

classification.loc[
    classification["competition_type"].eq("review")
].to_csv(
    output_dir / "competitions_to_review.csv",
    index=False,
)

print("Saved classification and classified matches.")

print(
    "Unreviewed games retained:",
    classified_matches["competition_type"].eq("review").sum(),
)

Saved classification and classified matches.
Unreviewed games retained: 43160


In [17]:
INITIAL_LOCAL_ELO = 1500.0
LOCAL_K = 24.0
LOCAL_ELO_SCALE = 400.0

required_regional_columns = [
    "match_id",
    "date",
    "league",
    "team_a",
    "team_b",
    "result_a",
    "competition_type",
    "rating_pool",
    "updates_home_pool",
    "use_local_elo",
    "use_region_calibration",
    "use_international_model",
]

missing_columns = (
    set(required_regional_columns)
    - set(classified_matches.columns)
)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

if classified_matches.empty:
    raise ValueError("No classified matches available.")

if not classified_matches["match_id"].is_unique:
    raise ValueError("Duplicate match IDs found.")

if classified_matches["date"].isna().any():
    raise ValueError("Missing timestamps found.")

if not classified_matches["result_a"].isin([0, 1]).all():
    raise ValueError("Results must be 0 or 1.")

for column in boolean_columns:
    if not classified_matches[column].isin([True, False]).all():
        raise ValueError(
            f"{column} must contain actual boolean values."
        )

print("Regional input checks passed.")

Regional input checks passed.


In [18]:
def local_expected_score(rating_a, rating_b):
    exponent = np.clip(
        (rating_b - rating_a) / LOCAL_ELO_SCALE,
        -300,
        300,
    )

    return 1.0 / (1.0 + 10.0 ** exponent)


assert np.isclose(local_expected_score(1500, 1500), 0.5)

assert np.isclose(
    local_expected_score(1600, 1500)
    + local_expected_score(1500, 1600),
    1.0,
)

print("Local Elo probability checks passed.")

Local Elo probability checks passed.


In [19]:
def generate_regional_features(
    match_table,
    initial_elo=INITIAL_LOCAL_ELO,
    k=LOCAL_K,
):
    local_ratings = {}
    local_game_counts = {}
    historical_homes = {}

    records = []

    ordered = match_table.sort_values(["date", "match_id"])

    for timestamp, batch in ordered.groupby("date", sort=True):
        pending_rating_changes = {}
        pending_game_counts = {}
        pending_home_updates = {}

        for row in batch.itertuples(index=False):
            # Domestic participation identifies the pool for this match.
            # Otherwise, use only previously established membership.
            if row.updates_home_pool:
                if pd.isna(row.rating_pool):
                    raise ValueError(
                        f"Missing domestic pool for {row.match_id}"
                    )

                pool_a = row.rating_pool
                pool_b = row.rating_pool

                for team in [row.team_a, row.team_b]:
                    previous_assignment = pending_home_updates.get(team)

                    if (
                        previous_assignment is not None
                        and previous_assignment != row.rating_pool
                    ):
                        raise ValueError(
                            f"{team} appears in different domestic pools "
                            f"at the same timestamp: {timestamp}"
                        )

                    pending_home_updates[team] = row.rating_pool

            else:
                pool_a = historical_homes.get(row.team_a)
                pool_b = historical_homes.get(row.team_b)

            known_a = pool_a is not None
            known_b = pool_b is not None

            key_a = (pool_a, row.team_a) if known_a else None
            key_b = (pool_b, row.team_b) if known_b else None

            pre_local_a = (
                local_ratings.get(key_a, initial_elo)
                if known_a
                else np.nan
            )

            pre_local_b = (
                local_ratings.get(key_b, initial_elo)
                if known_b
                else np.nan
            )

            games_a = (
                local_game_counts.get(key_a, 0)
                if known_a
                else 0
            )

            games_b = (
                local_game_counts.get(key_b, 0)
                if known_b
                else 0
            )

            record = row._asdict()

            record.update(
                {
                    "home_pool_a": pool_a,
                    "home_pool_b": pool_b,
                    "home_known_a": known_a,
                    "home_known_b": known_b,
                    "pre_local_elo_a": pre_local_a,
                    "pre_local_elo_b": pre_local_b,
                    "local_elo_diff": pre_local_a - pre_local_b,
                    "pre_local_games_a": games_a,
                    "pre_local_games_b": games_b,
                }
            )

            records.append(record)

            # Only eligible domestic matches change local Elo.
            if row.use_local_elo:
                if (
                    not row.updates_home_pool
                    or not known_a
                    or not known_b
                    or pool_a != pool_b
                ):
                    raise ValueError(
                        f"Invalid local-Elo configuration: {row.match_id}"
                    )

                probability_a = local_expected_score(
                    pre_local_a,
                    pre_local_b,
                )

                delta = k * (row.result_a - probability_a)

                pending_rating_changes[key_a] = (
                    pending_rating_changes.get(key_a, 0.0)
                    + delta
                )

                pending_rating_changes[key_b] = (
                    pending_rating_changes.get(key_b, 0.0)
                    - delta
                )

                for key in [key_a, key_b]:
                    pending_game_counts[key] = (
                        pending_game_counts.get(key, 0) + 1
                    )

        # Apply results after every prediction at this timestamp.
        for key, change in pending_rating_changes.items():
            local_ratings[key] = (
                local_ratings.get(key, initial_elo) + change
            )

        for key, count in pending_game_counts.items():
            local_game_counts[key] = (
                local_game_counts.get(key, 0) + count
            )

        historical_homes.update(pending_home_updates)

    return (
        pd.DataFrame(records),
        local_ratings,
        local_game_counts,
        historical_homes,
    )

In [20]:
example = pd.DataFrame(
    {
        "match_id": ["example_0", "example_1", "example_2", "example_3"],
        "date": pd.to_datetime(
            [
                "2020-01-01",
                "2020-01-01",
                "2020-01-02",
                "2020-01-03",
            ],
            utc=True,
        ),
        "league": ["WLDs", "Example R1", "Example R2", "WLDs"],
        "team_a": ["A", "A", "C", "A"],
        "team_b": ["C", "B", "D", "C"],
        "result_a": [1, 1, 0, 1],
        "competition_type": [
            "international",
            "domestic",
            "domestic",
            "international",
        ],
        "rating_pool": [None, "R1", "R2", None],
        "updates_home_pool": [False, True, True, False],
        "use_local_elo": [False, True, True, False],
        "use_region_calibration": [True, False, False, True],
        "use_international_model": [True, False, False, True],
    }
)

example_features, example_ratings, _, _ = (
    generate_regional_features(example)
)

example_features = example_features.set_index("match_id")

# No historical domestic membership exists at the first timestamp.
assert not example_features.loc["example_0", "home_known_a"]
assert not example_features.loc["example_0", "home_known_b"]

# First domestic appearances start at 1500.
assert np.isclose(
    example_features.loc["example_1", "pre_local_elo_a"],
    1500,
)

# Later international features use earlier domestic results.
assert example_features.loc["example_3", "home_pool_a"] == "R1"
assert example_features.loc["example_3", "home_pool_b"] == "R2"

assert np.isclose(
    example_features.loc["example_3", "pre_local_elo_a"],
    1512,
)

assert np.isclose(
    example_features.loc["example_3", "pre_local_elo_b"],
    1488,
)

# International results did not update local Elo.
assert np.isclose(example_ratings[("R1", "A")], 1512)
assert np.isclose(example_ratings[("R2", "C")], 1488)

# A future domestic result cannot alter an earlier prediction.
changed_example = example.copy()
changed_example.loc[
    changed_example["match_id"].eq("example_1"),
    "result_a",
] = 0

changed_features, _, _, _ = generate_regional_features(changed_example)
changed_features = changed_features.set_index("match_id")

assert not changed_features.loc["example_0", "home_known_a"]

assert pd.isna(
    changed_features.loc["example_0", "pre_local_elo_a"]
)

print("Regional chronology checks passed.")

Regional chronology checks passed.


In [21]:
(
    regional_features,
    final_local_ratings,
    final_local_game_counts,
    final_home_pools,
) = generate_regional_features(classified_matches)

assert len(regional_features) == len(classified_matches)
assert regional_features["match_id"].is_unique
assert regional_features["date"].is_monotonic_increasing

domestic_rows = regional_features.loc[
    regional_features["use_local_elo"]
]

assert domestic_rows[
    ["pre_local_elo_a", "pre_local_elo_b", "local_elo_diff"]
].notna().all().all()

assert domestic_rows["home_pool_a"].eq(
    domestic_rows["home_pool_b"]
).all()

print("Regional feature rows:", len(regional_features))

display(
    regional_features[
        [
            "date",
            "league",
            "team_a",
            "team_b",
            "home_pool_a",
            "home_pool_b",
            "pre_local_elo_a",
            "pre_local_elo_b",
            "local_elo_diff",
        ]
    ].head(10)
)

Regional feature rows: 96075


,date,league,team_a,team_b,home_pool_a,home_pool_b,pre_local_elo_a,pre_local_elo_b,local_elo_diff
0,2017-01-10 15:44:06+00:00,EU CS,EURONICS Gaming,Fnatic Academy,NaN,NaN,NaN,NaN,NaN
1,2017-01-10 16:21:23+00:00,EU CS,AlienTech eSports,Team Larssen,NaN,NaN,NaN,NaN,NaN
2,2017-01-10 17:01:50+00:00,EU CS,Bobaski,Nerv,NaN,NaN,NaN,NaN,NaN
3,2017-01-10 17:55:36+00:00,EU CS,Team Forge,Tricked Esport,NaN,NaN,NaN,NaN,NaN
4,2017-01-10 18:57:51+00:00,EU CS,ASUS ROG Army,LDLC OL,NaN,NaN,NaN,NaN,NaN
5,2017-01-11 15:33:10+00:00,EU CS,Fnatic Academy,LDLC OL,NaN,NaN,NaN,NaN,NaN
6,2017-01-11 15:46:32+00:00,EU CS,ALTERNATE aTTaX,AlienTech eSports,NaN,NaN,NaN,NaN,NaN
7,2017-01-11 16:44:20+00:00,EU CS,ASUS ROG Army,Nerv,NaN,NaN,NaN,NaN,NaN
8,2017-01-11 17:53:03+00:00,EU CS,Tricked Esport,devils.one inStreamly,NaN,NaN,NaN,NaN,NaN
9,2017-01-11 18:59:33+00:00,EU CS,Bobaski,EURONICS Gaming,NaN,NaN,NaN,NaN,NaN


In [22]:
international_rows = regional_features.loc[
    regional_features["use_international_model"]
].copy()

international_rows["both_homes_known"] = (
    international_rows["home_known_a"]
    & international_rows["home_known_b"]
)

home_coverage = (
    international_rows.groupby("league")
    .agg(
        games=("match_id", "size"),
        both_homes_known=("both_homes_known", "sum"),
    )
)

home_coverage["known_home_share"] = (
    home_coverage["both_homes_known"] / home_coverage["games"]
)

display(home_coverage)

unknown_home_matches = international_rows.loc[
    ~international_rows["both_homes_known"],
    [
        "date",
        "league",
        "team_a",
        "team_b",
        "home_pool_a",
        "home_pool_b",
    ],
]

print("International games with an unknown home:", len(unknown_home_matches))
display(unknown_home_matches.head(20))

,games,both_homes_known,known_home_share
league,,,
FST,80,80,1.000000
MSI,696,647,0.929598
WLDs,1176,1133,0.963435


International games with an unknown home: 92


,date,league,team_a,team_b,home_pool_a,home_pool_b
2552,2017-04-29 17:55:15+00:00,MSI,Lyon Gaming,Virtus.pro,NaN,LCL
2553,2017-04-29 19:26:17+00:00,MSI,GAM Esports,Isurus,NaN,NaN
2554,2017-04-29 20:33:35+00:00,MSI,Isurus,Virtus.pro,NaN,LCL
2555,2017-04-29 21:30:33+00:00,MSI,GAM Esports,Lyon Gaming,NaN,NaN
2556,2017-04-29 22:46:58+00:00,MSI,Isurus,Lyon Gaming,NaN,NaN
2557,2017-04-29 23:39:36+00:00,MSI,GAM Esports,Virtus.pro,NaN,LCL
2564,2017-05-01 18:00:05+00:00,MSI,Isurus,Virtus.pro,NaN,LCL
2565,2017-05-01 19:09:45+00:00,MSI,GAM Esports,Lyon Gaming,NaN,NaN
2566,2017-05-01 19:59:53+00:00,MSI,GAM Esports,Isurus,NaN,NaN
2567,2017-05-01 20:51:31+00:00,MSI,Lyon Gaming,Virtus.pro,NaN,LCL


In [23]:
local_rating_rows = [
    {
        "rating_pool": pool,
        "team": team,
        "local_elo": rating,
        "domestic_games": final_local_game_counts.get((pool, team), 0),
    }
    for (pool, team), rating in final_local_ratings.items()
]

local_ratings_table = pd.DataFrame(
    local_rating_rows,
    columns=["rating_pool", "team", "local_elo", "domestic_games"],
)

if local_ratings_table.empty:
    raise ValueError("No local ratings were generated.")

pool_checks = (
    local_ratings_table.groupby("rating_pool")
    .agg(
        teams=("team", "size"),
        total_rating=("local_elo", "sum"),
    )
)

pool_checks["expected_total"] = (
    pool_checks["teams"] * INITIAL_LOCAL_ELO
)

np.testing.assert_allclose(
    pool_checks["total_rating"],
    pool_checks["expected_total"],
)

print("Within-pool rating conservation passed.")
display(pool_checks)

display(
    local_ratings_table.sort_values(
        ["rating_pool", "local_elo"],
        ascending=[True, False],
    ).head(30)
)

Within-pool rating conservation passed.


,teams,total_rating,expected_total
rating_pool,,,
CBLOL,35,52500.0,52500.0
LCK,30,45000.0,45000.0
LCL,22,33000.0,33000.0
LCO,15,22500.0,22500.0
LCP,15,22500.0,22500.0
LCS,26,39000.0,39000.0
LEC,30,45000.0,45000.0
LJL,51,76500.0,76500.0
LLA,17,25500.0,25500.0


,rating_pool,team,local_elo,domestic_games
299,CBLOL,LØS,1758.229646,191
225,CBLOL,FURIA,1737.275851,323
256,CBLOL,LOUD,1664.812691,292
67,CBLOL,RED Canids,1618.148501,400
259,CBLOL,Vorax Liberty,1602.587005,48
224,CBLOL,Prodigy Esports,1565.863524,47
244,CBLOL,Santos e-Sports,1538.218444,21
483,CBLOL,KaBuM! Ilha das Lendas,1537.244452,18
46,CBLOL,paiN Gaming,1535.252501,443
306,CBLOL,Vivo Keyd Stars,1529.826926,179


In [24]:
regional_features.to_csv(
    output_dir / "regional_elo_features_all_years.csv",
    index=False,
)

local_ratings_table.to_csv(
    output_dir / "local_elo_ratings_all_years.csv",
    index=False,
)

home_pool_table = pd.DataFrame(
    [
        {"team": team, "latest_home_pool": pool}
        for team, pool in final_home_pools.items()
    ],
    columns=["team", "latest_home_pool"],
)

home_pool_table.to_csv(
    output_dir / "latest_team_home_pools.csv",
    index=False,
)

unknown_home_matches.to_csv(
    output_dir / "international_unknown_home_matches.csv",
    index=False,
)

print("Regional outputs saved to:", output_dir)

Regional outputs saved to: c:\Users\USER\lol_worlds_predictor\data\processed
